# Variational Warmstart Ablation
**Tommaso R. Marena (2026) — Section 5.3**

Measures the speedup from variational warmstart (Contribution 3) on all six real datasets.
For each dataset, records M_cold (samples without warmstart), M_warm (with warmstart),
and the speedup ratio. Produces the per-dataset ablation table for Section 5.3.

**Produces:**
- `warmstart_ablation_table.json` — M_cold, M_warm, speedup per dataset
- `warmstart_ablation.pdf` — bar chart of speedup by dataset
- LaTeX table snippet

> **Runtime:** ~60 min on T4.

In [ ]:
import subprocess, sys

# Step 1: install qos package wheel only (--no-deps) so we don't touch
# Colab's pre-pinned numpy/scipy/jax, which would cause version conflicts.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '--force-reinstall', '--no-deps',
    'quantum-oracle-sketching @ git+https://github.com/Tommaso-R-Marena/quantum_oracle_sketching.git'
], check=True)

# Step 2: install qos's non-numeric Python deps that Colab doesn't provide.
# Explicitly exclude numpy/scipy/jax to avoid downgrading Colab's versions.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'optax', 'flax', 'qiskit', 'pyqsp'
], check=True)

# Step 3: notebook-only extras.
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'scanpy', 'anndata', 'leidenalg', 'python-igraph'
], check=True)

print('Installed.')

In [ ]:
import os
MOUNT_DRIVE = True
if MOUNT_DRIVE:
    from google.colab import drive; drive.mount('/content/drive')
    OUTPUT_DIR = '/content/drive/MyDrive/qos_warmstart'
else:
    OUTPUT_DIR = '/content/qos_warmstart'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
import jax
jax.config.update('jax_enable_x64', True)
import jax.numpy as jnp
import numpy as np, json, time, warnings
import matplotlib.pyplot as plt, matplotlib
matplotlib.rcParams.update({'figure.dpi': 150, 'font.size': 10, 'font.family': 'serif'})

EPSILON_TARGET  = 0.05
M_MAX           = 50000
NUM_FOURIER     = 64
WARMSTART_STEPS = 400
WARMSTART_LR    = 0.03
NUM_TRIALS      = 3
SEED            = 42
print(f'JAX x64 enabled: {jax.config.jax_enable_x64}')
print(f'Epsilon={EPSILON_TARGET}, M_max={M_MAX}')
print(f'Warmstart: {NUM_FOURIER} modes, {WARMSTART_STEPS} steps, lr={WARMSTART_LR}')

## 1. Helpers

In [ ]:
import scipy.sparse as sp
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from qos.core.oracle_sketch import q_oracle_sketch_boolean
from qos.theory import VariationalWarmstart

def tvd_diag(diag_approx, diag_ideal):
    """Total variation distance between two oracle diagonals.

    Both inputs should be real-valued +-1 arrays.
    Complex input (e.g. from VariationalWarmstart.predict()) is handled
    by taking the real part before computing probabilities.
    """
    N = len(diag_ideal)
    n = int(np.log2(N))
    H = np.array([[1,1],[1,-1]])/np.sqrt(2)
    Hn = H.copy()
    for _ in range(n-1): Hn = np.kron(Hn, H)
    def probs(d):
        d_arr = np.real(np.array(d, dtype=np.complex128)).astype(np.float64)
        s = Hn @ (d_arr / np.sqrt(N))
        p = np.abs(s)**2; return p/p.sum()
    return 0.5*float(np.sum(np.abs(probs(diag_approx)-probs(diag_ideal))))

def find_M_cold(truth_table, epsilon=EPSILON_TARGET, m_max=M_MAX):
    tt = jnp.array(truth_table, dtype=jnp.float64)
    d_ideal = (-1.0) ** tt
    lo, hi = 10, m_max
    while lo < hi - 1:
        mid = (lo + hi) // 2
        d, _ = q_oracle_sketch_boolean(tt, mid)
        if tvd_diag(d, d_ideal) < epsilon: hi = mid
        else: lo = mid
    return hi

def find_M_warm(truth_table, epsilon=EPSILON_TARGET, m_max=M_MAX, trial_seed=SEED):
    """Honest ablation: gate truth table to exactly `mid` oracle queries.

    VariationalWarmstart only sees `mid` randomly sampled entries
    (zeros elsewhere), preventing information leakage from the full table.
    """
    tt = jnp.array(truth_table, dtype=jnp.float64)
    N = int(tt.shape[0])
    d_ideal = (-1.0) ** tt
    rng = np.random.default_rng(trial_seed)
    lo, hi = 10, m_max
    while lo < hi - 1:
        mid = (lo + hi) // 2
        n_queries = min(mid, N)
        idx = rng.choice(N, size=n_queries, replace=False)
        tt_sub = jnp.zeros(N, dtype=jnp.float64).at[idx].set(tt[idx])
        vw = VariationalWarmstart(tt_sub, num_fourier_modes=NUM_FOURIER,
                                  learning_rate=WARMSTART_LR, num_steps=WARMSTART_STEPS,
                                  key=jax.random.PRNGKey(trial_seed))
        vw.fit(unit_num_samples=mid * 4)
        d_warm = jnp.sign(jnp.real(vw.predict()))
        if tvd_diag(d_warm, d_ideal) < epsilon: hi = mid
        else: lo = mid
    return hi

def diagnose_warmstart(truth_table, trial_seed=SEED):
    """Convergence gate: verify warmstart achieves TVD < epsilon at M_MAX budget.

    Uses unit_num_samples=M_MAX*4 to match the budget scale used by
    find_M_warm at the top of its binary search (mid ~ M_MAX/2).
    """
    tt = jnp.array(truth_table, dtype=jnp.float64)
    d_ideal = (-1.0) ** tt
    vw = VariationalWarmstart(tt, num_fourier_modes=NUM_FOURIER,
                              learning_rate=WARMSTART_LR, num_steps=WARMSTART_STEPS,
                              key=jax.random.PRNGKey(trial_seed))
    vw.fit(unit_num_samples=M_MAX * 4)
    d_warm = jnp.sign(jnp.real(vw.predict()))
    tvd = tvd_diag(d_warm, d_ideal)
    losses = vw.convergence_losses
    print(f'  [diag] TVD at M_MAX budget: {tvd:.4f}  (target < {EPSILON_TARGET})')
    print(f'  [diag] Loss: {losses[0]:.4f} -> {losses[-1]:.4f}  (converged={losses[-1] < losses[0]})')
    return tvd

print('Helpers loaded.')

## 2. Load Datasets & Build Truth Tables

In [ ]:
from sklearn.datasets import fetch_20newsgroups

def pc1_truth_table(X, N=256):
    if sp.issparse(X): X = X.toarray()
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    svd = TruncatedSVD(n_components=1, random_state=SEED)
    pc = svd.fit_transform(X).ravel()
    pc = np.resize(pc, N)
    return (pc >= np.median(pc)).astype(float)

datasets = {}
N_TT = 256

print('Loading IMDb...')
from qos.experiments.real_datasets.imdb.imdb_utils import load_imdb_data
X_raw, _ = load_imdb_data()
vec = TfidfVectorizer(min_df=10, stop_words='english', max_features=N_TT)
X_imdb = vec.fit_transform(X_raw[:5000])
datasets['IMDb'] = pc1_truth_table(X_imdb, N_TT)

print('Loading 20 Newsgroups...')
data = fetch_20newsgroups(subset='train', remove=('headers','footers','quotes'))
vec2 = TfidfVectorizer(min_df=10, stop_words='english', max_features=N_TT)
X_news = vec2.fit_transform(data.data[:5000])
datasets['20 Newsgroups'] = pc1_truth_table(X_news, N_TT)

print('Loading PBMC3k...')
from qos.experiments.real_datasets.pbmc68k.pbmc_utils import load_pbmc3k
adata3k, _ = load_pbmc3k()
X3k = adata3k.X.toarray() if sp.issparse(adata3k.X) else adata3k.X
datasets['PBMC3k'] = pc1_truth_table(X3k[:, :N_TT], N_TT)

print('Loading PBMC68k (reduced)...')
import scanpy as sc
adata68k = sc.datasets.pbmc68k_reduced()
sc.pp.filter_cells(adata68k, min_counts=1)
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    sc.pp.normalize_total(adata68k, target_sum=1e4)
    sc.pp.log1p(adata68k)
X68k = adata68k.X.toarray() if sp.issparse(adata68k.X) else adata68k.X
datasets['PBMC68k'] = pc1_truth_table(X68k[:, :N_TT], N_TT)

print('Loading Dorothea...')
from qos.experiments.real_datasets.dorothea.dorothea_utils import load_dorothea
X_dor, _ = load_dorothea()
datasets['Dorothea'] = pc1_truth_table(X_dor, N_TT)

print('Loading Splice...')
from qos.experiments.real_datasets.splice.splice_utils import load_splice
X_spl, _ = load_splice()
datasets['Splice'] = pc1_truth_table(X_spl, N_TT)

print(f'Loaded {len(datasets)} datasets, N={N_TT} each.')

## 2b. Sanity Check — Warmstart Convergence at Full Budget
Run this before the full ablation. If TVD > 0.05 here, increase NUM_FOURIER or WARMSTART_STEPS in the config cell.

In [ ]:
print('Warmstart convergence diagnostics (M_MAX budget, first dataset only):')
name0, tt0 = next(iter(datasets.items()))
print(f'{name0}:')
tvd0 = diagnose_warmstart(tt0)
assert tvd0 < EPSILON_TARGET, (
    f'Warmstart does not converge at M_MAX budget (TVD={tvd0:.4f} >= {EPSILON_TARGET}). '
    'Increase NUM_FOURIER, WARMSTART_STEPS, or lower WARMSTART_LR in the config cell.'
)
print('Convergence OK — proceeding to ablation.')

## 3. Run Ablation

In [ ]:
ablation_results = {}
t0 = time.time()
rng_ab = np.random.default_rng(SEED)

for name, base_tt in datasets.items():
    print(f'\n{name}:')
    m_colds, m_warms = [], []
    for trial in range(NUM_TRIALS):
        noise_mask = rng_ab.random(len(base_tt)) < 0.05
        tt = base_tt.copy(); tt[noise_mask] = 1 - tt[noise_mask]
        mc = find_M_cold(tt)
        mw = find_M_warm(tt, trial_seed=SEED + trial)
        m_colds.append(mc); m_warms.append(mw)
        print(f'  trial {trial+1}: M_cold={mc}, M_warm={mw}, speedup={mc/max(mw,1):.2f}x')
    ablation_results[name] = {
        'M_cold_mean':  float(np.mean(m_colds)),
        'M_warm_mean':  float(np.mean(m_warms)),
        'M_cold_std':   float(np.std(m_colds)),
        'M_warm_std':   float(np.std(m_warms)),
        'speedup_mean': float(np.mean(m_colds) / max(np.mean(m_warms), 1)),
        'epsilon':      EPSILON_TARGET,
    }
    sp_str = f"{ablation_results[name]['speedup_mean']:.2f}"
    print(f'  => M_cold={np.mean(m_colds):.0f}+/-{np.std(m_colds):.0f}  M_warm={np.mean(m_warms):.0f}+/-{np.std(m_warms):.0f}  speedup={sp_str}x')
    with open(os.path.join(OUTPUT_DIR,'warmstart_ablation_table.json'),'w') as f:
        json.dump(ablation_results, f, indent=2)

print(f'\nAll done. Total time: {time.time()-t0:.0f}s')

In [ ]:
names    = list(ablation_results.keys())
speedups = [ablation_results[n]['speedup_mean'] for n in names]
m_cold   = [ablation_results[n]['M_cold_mean'] for n in names]
m_warm   = [ablation_results[n]['M_warm_mean'] for n in names]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
x = np.arange(len(names))
ax1.bar(x-0.2, m_cold, 0.38, label='M_cold (no warmstart)', color='tomato', alpha=0.85)
ax1.bar(x+0.2, m_warm, 0.38, label='M_warm (with warmstart)', color='seagreen', alpha=0.85)
ax1.set_xticks(x); ax1.set_xticklabels(names, rotation=20, ha='right')
ax1.set_ylabel('Samples to reach TVD < 0.05')
ax1.set_title('Sample budget: cold vs warm start')
ax1.legend(); ax1.grid(True, alpha=0.3, axis='y')

bars = ax2.bar(names, speedups, color='steelblue', alpha=0.85)
ax2.axhline(1.0, color='k', ls='--', lw=1)
for bar, s in zip(bars, speedups):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
             f'{s:.2f}x', ha='center', va='bottom', fontsize=9)
ax2.set_ylabel('Speedup (M_cold/M_warm)')
ax2.set_title('Warmstart speedup by dataset')
ax2.set_xticks(np.arange(len(names))); ax2.set_xticklabels(names, rotation=20, ha='right')
ax2.grid(True, alpha=0.3, axis='y')

plt.suptitle(f'Variational Warmstart Ablation (Marena 2026, epsilon={EPSILON_TARGET})', fontsize=11)
plt.tight_layout()
path = os.path.join(OUTPUT_DIR,'warmstart_ablation.pdf')
plt.savefig(path, bbox_inches='tight'); plt.show()
print(f'Saved: {path}')

print('\n--- LaTeX table (Section 5.3) ---')
rows = [r'\begin{tabular}{lccc}', r'\toprule',
        r'Dataset & $M_{\mathrm{cold}}$ & $M_{\mathrm{warm}}$ & Speedup \\', r'\midrule']
for nm in names:
    r = ablation_results[nm]
    rows.append(f"{nm} & {r['M_cold_mean']:.0f}$\\pm${r['M_cold_std']:.0f} & {r['M_warm_mean']:.0f}$\\pm${r['M_warm_std']:.0f} & {r['speedup_mean']:.2f}x \\\\")
rows += [r'\bottomrule', r'\end{tabular}']
print('\n'.join(rows))